# Aayvu local open-weight code reviewer
Required: Colab Pro, Python 3, **G4 GPU**, **High-RAM On**. Publication defaults off and requires confirmation.

In [ ]:
# 1. Runtime settings
RUN_MODE = "pending"  # pending | explicit
PUBLISH = False
EXPLICIT_REPOSITORY = "SurveyBuilderLLM/backend"
EXPLICIT_PR = None
PROJECT_GIT_URL = "https://github.com/sivab193/self-reviewer.git"
PROJECT_REF = "b78d267cc2ec4b3530cb71ccf5c9bef43d827961"  # immutable reviewed source revision


In [ ]:
# 2. Load secrets and install the pinned reviewer and OCR
from google.colab import userdata
import os, re, subprocess, tempfile
os.environ['GITHUB_REVIEW_TOKEN'] = userdata.get('GITHUB_REVIEW_TOKEN')
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
assert re.fullmatch(r'[0-9a-f]{40}', PROJECT_REF), 'Pin PROJECT_REF to a commit SHA'
with tempfile.TemporaryDirectory() as td:
    askpass = os.path.join(td, 'askpass.sh')
    open(askpass, 'w').write('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) printf "%s" "$GITHUB_REVIEW_TOKEN";; esac\n')
    os.chmod(askpass, 0o700)
    clone_env = {**os.environ, 'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0'}
    subprocess.run(['git','clone','--filter=blob:none',PROJECT_GIT_URL,'/content/code-reviewer'], env=clone_env, check=True)
subprocess.run(['git','-C','/content/code-reviewer','checkout','--detach',PROJECT_REF], check=True)
subprocess.run(['pip','install','-e','/content/code-reviewer[colab]'], check=True)
subprocess.run(['npm','install','-g','@alibaba-group/open-code-review@1.13.0'], check=True)


In [ ]:
# 3. Enforce G4/High-RAM/disk/CUDA checks
from pathlib import Path
from aayvu_reviewer.config import load_model_profile
from aayvu_reviewer.hardware import detect, require_g4
profile = load_model_profile(Path('/content/code-reviewer/config/model-profile.yaml'))
hardware = detect(Path(profile.cache_dir)); require_g4(hardware)
hardware


In [ ]:
# 4. Start the loopback-only BF16 model server and run deterministic smoke test
from datetime import datetime, timezone
from aayvu_reviewer.model_server import ModelServer
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = Path('/content/code-reviewer/runs') / RUN_ID
RUN_DIR.mkdir(parents=True)
server = ModelServer(profile, RUN_DIR / 'vllm.log', os.environ.get('HF_TOKEN'))
server.start()


In [ ]:
# 5. Verify OCR can use the local endpoint
ocr_env = {**os.environ, 'OCR_LLM_URL': server.base_url + '/chat/completions', 'OCR_LLM_AUTH_TOKEN': 'local-colab-only', 'OCR_LLM_MODEL': profile.served_model_name, 'OCR_LLM_PROTOCOL': 'openai'}
subprocess.run(['ocr','llm','test'], env=ocr_env, check=True)


In [ ]:
# 6. Discover and preview; no GPU review or GitHub writes happen here
args = ['aayvu-reviewer','discover']
subprocess.run(args, cwd='/content/code-reviewer', check=True)


## Publication gate
Inspect the preview above. `PUBLISH=False` performs a full review but no GitHub writes. Setting it true still prompts for confirmation.

In [ ]:
# 7–9. Review sequentially and report artifact location
args = ['aayvu-reviewer','review']
if RUN_MODE == 'pending': args += ['--pending']
else:
    assert EXPLICIT_PR is not None
    args += ['--repo', EXPLICIT_REPOSITORY, '--pr', str(EXPLICIT_PR)]
args += ['--publish' if PUBLISH else '--dry-run', '--run-root', '/content/code-reviewer/runs']
subprocess.run(args, cwd='/content/code-reviewer', env=ocr_env, check=True)


In [ ]:
# 10. Stop model server and clear secret-bearing process environment
server.stop()
os.environ.pop('GITHUB_REVIEW_TOKEN', None)
os.environ.pop('HF_TOKEN', None)
print('Stopped. Artifacts are under /content/code-reviewer/runs')
